# Eagle5 v2 Corpus Build (Colab — minimal)

5 cells. No patches, no version pinning, no per-tier branching. Just:
1. Mount Drive
2. Install deps
3. Clone repo
4. Run `colab/corpus_simple.py` (~150 lines, single-layer capture, aggressive memory cleanup)
5. Sync local → Drive

Run cells 1→5 in order. Expected wall: ~30-60 min on Blackwell/A100, ~1.5 hr on L4, ~2 hr on V100/T4.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No CUDA — set Runtime → Change runtime type → GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')

In [ ]:
# Cell 2 — Install / verify deps (most pre-installed on Colab)
!pip install -q 'accelerate>=1.0' 'datasets>=3.0' 'pyarrow>=17' 'tqdm>=4.66' 'zstandard'
import transformers, datasets, pyarrow, accelerate
print(f'transformers={transformers.__version__}  datasets={datasets.__version__}')
print(f'pyarrow={pyarrow.__version__}  accelerate={accelerate.__version__}')

In [ ]:
# Cell 3 — Clone dismantle (we only need colab/corpus_simple.py)
import os
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls -l colab/corpus_simple.py

In [ ]:
# Cell 4 — RUN CORPUS BUILD
#
# Defaults: 100k seqs × 2048 max-tokens × shard-size=16. Captures layer
# 25 only (residual + intermediate) — what eagle5_train.py reads.
# Output: /content/v2_lite_corpus (local SSD, fast).
#
# Resumable: re-run after any interruption — corpus_simple.py detects
# existing shards and continues from the next one. Aggressive
# torch.cuda.empty_cache() per batch.

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if   vram >= 70: batch = 16
elif vram >= 35: batch = 12
elif vram >= 20: batch = 8
else:            batch = 4
print(f'using batch={batch} (VRAM {vram:.0f} GB)')

!python colab/corpus_simple.py \
  --out /content/v2_lite_corpus \
  --max-sequences 100000 \
  --max-tokens 2048 \
  --batch-size {batch} \
  --capture-layer 25 \
  --shard-size 16

In [ ]:
# Cell 5 — Sync local corpus → Drive (so it survives VM teardown)
import os, glob, shutil, time
SRC = '/content/v2_lite_corpus'
DST = '/content/drive/MyDrive/dismantle/v2_lite_corpus'
os.makedirs(DST, exist_ok=True)

src_shards = sorted(glob.glob(f'{SRC}/*.parquet'))
if not src_shards:
    print(f'⚠️  no shards at {SRC} — Cell 4 not complete?')
else:
    print(f'syncing {len(src_shards)} shards → Drive (resumable)…')
    t0 = time.time()
    for i, s in enumerate(src_shards):
        d = f'{DST}/{os.path.basename(s)}'
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
            shutil.copy2(s, d)
        if (i+1) % 100 == 0 or i == len(src_shards)-1:
            print(f'  {i+1}/{len(src_shards)} ({time.time()-t0:.0f}s)')
    total_mb = sum(os.path.getsize(f) for f in glob.glob(f'{DST}/*.parquet')) / 1e6
    print(f'\n✅ {len(src_shards)} shards on Drive, {total_mb:.0f} MB')
    print(f'\nNext: open Drive → MyDrive/dismantle/v2_lite_corpus → Download (zip)')
    print(f'Then on laptop: unzip into artifacts/calibration/v2_lite_corpus/')